# DDR/DHL Assignment 3: Linked Open Data
This notebook contains the third, collaborative, graded assignment of the 2026 Data-Driven Research/Digital Humanities Lab course. In this assignment you'll use Linked Open Data tools in order to search for information on the Web in a more thorough way than with Google.

To complete the assignment, complete the highlighted **Part 1, Part 2, Part 3 and Part 4**.

This is a collaborative assignment. In the text cell below, please include all the names of your group members.

Any complete and working submission will be a passing grade. Note that although you will further develop your Python skills by doing the assignment, the learning objective is in the content. For high grades, we look in particular at reflection and understanding of power of semantic search, and its strengths and weaknesses, as well as the potential value for data-driven research.

Finally, if you used code or a solution from the internet (such as StackOverflow or Gemini/ChatGPT) or another external resource, please make reference to it (in any format) and make sure there is a clear explanation of the solution. We permit this as long as this is clearly disclosed in the assignment and you demonstrate to understand the final code. Unattributed copied code will be considered plagiarism and therefore fraud.

***ENSURE TO RUN ALL CELLS AND SUBMIT YOUR NOTEBOOK WITH ALL OUTPUT INCLUDED!***

**Authors of this answer:**

# 1. Introduction

In this exercise, you'll experiment with a very explicit approach to semantics, and experience how powerful a little semantics can be when searching.

You'll use the DBpedia knowledge base, essentially the content of Wikipedia in machine readable form, and explore what explicit semantic enables using [SPARQL](http://en.wikipedia.org/wiki/SPARQL). Queries allowing you to query and search through the Web of Linked Open Data. We will use the Python SPARQLWrapper library to access the DBpedia endpoint.

## 1.1. RDF
Linked Open Data consists of a huge number of small facts, in the form of RDF triples, <*Subject*, *Predicate*, *Object*>, which each consists of a pair of concepts or entities and a relationship between them, such as `Rembrandt birthPlace Leiden`. We work specifically with DBpedia's machine readable information from Wikipedia, in this case http://dbpedia.org/page/Rembrandt:
`dbr:Rembrandt dbo:birthPlace dbr:Leiden`

It's hard to guess upfront how information is encoded in DBpedia, and linked data is all about having unique identifiers for every entity or concept.   The best way is to look at examples, and use google to kickstart to find a particular DBpedia entity.

For example, Google "dbpedia rembrandt" which will give you a neat page with DBpedia facts about him (https://dbpedia.org/page/Rembrandt).   If you look at the link of the "About: Rembrandt" you find the unique link that is the identifier of this entity is http://dbpedia.org/resource/Rembrandt, and entering this link/ID in your browser will generate the overview page.

Inside DBpedia, you can use a shorthand  `dbr:Rembrandt` (which is defined to unfold to http://dbpedia.org/resource/Rembrandt) as the unique ID, but it also works if you use the long URL!  Recall that DBpedia does not have pages, but only countless RDF triples, and the overview page is just the output of all triples, <`dbr:Rembrandt` *Predicate*, *Object*>, allowing you also to see what further relations and concepts to explore.

## 1.2. SPARQL
SPARQL is the designated query language for RDF modelled on an extended SQL relational database query language.  It uses natural language words like SELECT, DISTINCT, WHERE, ORDER BY, and LIMIT in a very specific way. We introduce it by example, but feel free to backtrack to one of the many tutorial and introductions on the web.  

# 2. Working with SPARQL queries

There is a large database of facts derived from Wikipedia, called [DBpedia](http://dbpedia.org/About), which contains information about everything and the rest.  Let's look at 1990s American Films in the first part of this assignment. You will access a so-called SPARQL endpoint for DBpedia through Python. Each film in the category 1990s American Films has as a fact about it in DBpedia that is has a relationship with the category 1990s American Films, namely, the `dct:subject` property. That is, the film Pulp Fiction has an RDF triple
`dbr:Pulp_Fiction dct:subject category:1990s_American_films`

If you put the name of the entity `Pulp_Fiction`, so `dbr:Pulp_Fiction` which is shorthand for http://dbpedia.org/resource/Pulp_Fiction, your browser will generate a page http://dbpedia.org/page/Pulp_Fiction with a selection of facts  <`dbr:Pulp_Fiction`, ?, ?> in the data base.

Now, let's access this database through Python. We will need the SPARQLWrapper, which does not come with Google Colab by default, so we should install it:

In [2]:
#!pip install sparqlwrapper

Then, we are able to import it and choose the DBPedia endpoint to access that database:

In [3]:
from SPARQLWrapper import SPARQLWrapper
import pandas as pd
from io import StringIO
from IPython.display import display, HTML

# Specify the DBPedia endpoint
sparql = SPARQLWrapper("http://dbpedia.org/sparql")

# Specify that we want results in the CSV format
sparql.setReturnFormat('csv')

# Note that general DBPedia endpoint contains the short notation (or PREFIX) for many namespaces.
#    See https://dbpedia.org/sparql/?help=nsdecl for a list
#
# This allows you to write http://dbpedia.org/resource/Pulp_Fiction as dbr:Pulp_Fiction
#
# On other endpoints, you must declare these explicitly by adding to the SPARQL query:
# PREFIX dbr: <http://dbpedia.org/resource/>
# PREFIX dbo: <http://dbpedia.org/ontology/>
# PREFIX dct: <http://purl.org/dc/terms/>
# PREFIX dbc: <http://dbpedia.org/resource/Category:>
# PREFIX rdf:	<http://www.w3.org/1999/02/22-rdf-syntax-ns#>

Now, we can try some SPARQL queries. First we set the query as a multi-line string, and then we use the query function to actually run the query.

Note that in the query below, ?film is a variable name, just like a Python variable - we could rename it to anything else, and the result would be the same.

In [4]:
sparql.setQuery("""
    SELECT ?film
    WHERE {?film dct:subject dbc:1990s_American_films}
    LIMIT 1000
""")

result = sparql.query().convert().decode("utf-8")
print(result[:225])

"film"
"http://dbpedia.org/resource/Feed_(1992_film)"
"http://dbpedia.org/resource/Champagne_and_Bullets"
"http://dbpedia.org/resource/Undercurrent_(1998_film)"
"http://dbpedia.org/resource/White_Tiger_(1996_film)"
"http://db


The result variable now contains a list in CSV format of 1000 links to 1990s American films on dbpedia (we limited the number of results to 1000, but it can be increased). To make it easier to visualize, let's turn it into a Pandas dataframe (this will become more useful as we retrieve more properties):

In [5]:
df = pd.read_csv(StringIO(result), sep=",")
df

,film
0,http://dbpedia.org/resource/Feed_(1992_film)
1,http://dbpedia.org/resource/Champagne_and_Bullets
2,http://dbpedia.org/resource/Undercurrent_(1998...
3,http://dbpedia.org/resource/White_Tiger_(1996_...
4,http://dbpedia.org/resource/DNA_(1997_film)
...,...
995,http://dbpedia.org/resource/Starquest_II
996,http://dbpedia.org/resource/The_Killer_Eye
997,http://dbpedia.org/resource/Grind_(1997_film)
998,http://dbpedia.org/resource/Lee_Marvin:_A_Pers...


We see that the enitities, e.g. `dbc:1990s_American_films` are URI's referring to a unique entity in the linked open data cloud. So the name is a unique ID, in this case shorthand for http://dbpedia.org/resource/Category:1990s_American_films.  

Take a closer look at that SPARQL query. Can you figure out how it works? If you're familiar with structured query languages like SQL, you'll recognize many aspects. If this is completely new to you, there are still many recognizable words to help you interpret this query. Let's get back to this later.

To see the power of this form of searching, let's try a slightly more complex query, where you add a second RDF-like condition to the query, separated from the first by a dot `.` representing a `join` or `AND`:

In [6]:
sparql.setQuery("""
    SELECT ?film ?actor
    WHERE {?film dct:subject dbc:1990s_American_films . ?film dbo:starring ?actor }
    LIMIT 1000
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,film,actor
0,http://dbpedia.org/resource/Champagne_and_Bullets,http://dbpedia.org/resource/Pamela_Bryant
1,http://dbpedia.org/resource/Champagne_and_Bullets,http://dbpedia.org/resource/Wings_Hauser
2,http://dbpedia.org/resource/Undercurrent_(1998...,http://dbpedia.org/resource/Lorenzo_Lamas
3,http://dbpedia.org/resource/Undercurrent_(1998...,http://dbpedia.org/resource/Daniel_Lugo_(actor)
4,http://dbpedia.org/resource/Undercurrent_(1998...,http://dbpedia.org/resource/Brenda_Strong
...,...,...
995,http://dbpedia.org/resource/Deadly_Relations,http://dbpedia.org/resource/Robert_Urich
996,http://dbpedia.org/resource/Deadly_Voyage,http://dbpedia.org/resource/Sean_Pertwee
997,http://dbpedia.org/resource/Deadly_Voyage,http://dbpedia.org/resource/Adewale_Akinnuoye-...
998,http://dbpedia.org/resource/Deadly_Voyage,http://dbpedia.org/resource/Omar_Epps


You should now get a table with American films and the actors that have played in them. This list is not complete. The content of DBpedia is based on the Infoboxes of Wikipedia pages, which have a standard format. The knowledge in DBpedia is as good as the encyclopedic information on Wikipedia. Not all actors per film are listed and not all films and actors have their own Wikipedia article.

Try a few more SPARQL queries, given below. See if you can figure out before hand what results they will give:

In [7]:
sparql.setQuery("""
    SELECT DISTINCT ?actor
    WHERE {?film dct:subject dbc:1990s_American_films . ?film dbo:starring ?actor }
    LIMIT 1000
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,actor
0,http://dbpedia.org/resource/Pamela_Bryant
1,http://dbpedia.org/resource/Wings_Hauser
2,http://dbpedia.org/resource/Lorenzo_Lamas
3,http://dbpedia.org/resource/Daniel_Lugo_(actor)
4,http://dbpedia.org/resource/Brenda_Strong
...,...
995,http://dbpedia.org/resource/Matt_Mulhern
996,http://dbpedia.org/resource/Valerie_Landsburg
997,http://dbpedia.org/resource/Ellen_Burstyn
998,http://dbpedia.org/resource/Barnard_Hughes


In [8]:
sparql.setQuery("""
    SELECT (COUNT(?film) AS ?count) ?actor
    WHERE {?film dct:subject dbc:1990s_American_films . ?film dbo:starring ?actor }
    ORDER BY DESC(?count)
    LIMIT 1000
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,count,actor
0,29,http://dbpedia.org/resource/Christopher_Walken
1,29,http://dbpedia.org/resource/Harvey_Keitel
2,28,http://dbpedia.org/resource/Bill_Pullman
3,27,http://dbpedia.org/resource/Eric_Roberts
4,26,http://dbpedia.org/resource/Whoopi_Goldberg
...,...,...
995,5,http://dbpedia.org/resource/Dina_Meyer
996,5,http://dbpedia.org/resource/Meg_Foster
997,5,http://dbpedia.org/resource/John_Beck_(actor)
998,5,http://dbpedia.org/resource/Stacey_Travis


**Part 1**: Adapt the SPARQL query above to count per film how many actors it has. This requires a *minimal change* to the query.   

In [9]:
# Your adapted query here
sparql.setQuery("""
 SELECT (COUNT(?film) AS ?count) ?film
    WHERE {?film dct:subject dbc:1990s_American_films . ?film dbo:starring ?actor }
    ORDER BY DESC(?count)
    LIMIT 1000

""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,count,film
0,25,http://dbpedia.org/resource/Mother_Goose_Rock_...
1,25,http://dbpedia.org/resource/Prêt-à-Porter_(film)
2,22,http://dbpedia.org/resource/Short_Cuts
3,20,http://dbpedia.org/resource/Alien_Arsenal
4,19,http://dbpedia.org/resource/Deconstructing_Harry
...,...,...
995,6,http://dbpedia.org/resource/The_Ballad_of_Litt...
996,6,http://dbpedia.org/resource/Another_Midnight_Run
997,6,http://dbpedia.org/resource/Street_Fighter_(19...
998,6,http://dbpedia.org/resource/Cutthroat_Island


# 3. Six Degrees of Kevin Bacon

There is a game related to the notion of [Six Degrees of Separation](http://en.wikipedia.org/wiki/Six_degrees_of_separation). This involves the network of actors who have played in a film together with Kevin Bacon, and actors who have played with those actors, etc. The goal is to figure out the shortest path, between actors who have co-starred in a film, between any Hollywood actor and the actor Kevin Bacon. One of the research aspects is whether the network of actors in Hollywood form a so-called [Small World](http://en.wikipedia.org/wiki/Small-world_experiment) network.

**Part 2** (some questions in the steps below)

Steps:

1. Let's first query:

In [10]:
sparql.setQuery("""
    SELECT ?film
    WHERE {?film dbo:starring dbr:Kevin_Bacon}
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,film
0,http://dbpedia.org/resource/The_Toxic_Avenger_...
1,http://dbpedia.org/resource/Beverly_Hills_Cop:...
2,http://dbpedia.org/resource/Footloose
3,http://dbpedia.org/resource/Loverboy_(2005_film)
4,http://dbpedia.org/resource/Hollow_Man
...,...
69,http://dbpedia.org/resource/100_Greatest_Artis...
70,http://dbpedia.org/resource/Murder_in_the_Firs...
71,http://dbpedia.org/resource/Sleepers_(film)
72,http://dbpedia.org/resource/The_Air_Up_There


2. How many films are in the list? We can count using a COUNT statement:

In [11]:
sparql.setQuery("""
    SELECT COUNT(?film)
    WHERE {?film dbo:starring dbr:Kevin_Bacon}
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,callret-0
0,74


3. Is this list complete? Compare this number with, for instance, the number of films listed on Kevin Bacon's IMDB page.

**Answer:** Compared to the number of films listed on Kevin Bacon's IMDB page, this list is less complete. The current list consists of 74 films, whereas the IMDB page consists of 113 films. 

4. To see the power of this form of searching, let's gradually make this into a more complex query. Let's ask for the list of actors co-starring with Kevin Bacon. You can add a second RDF-like condition to the query, separated from the first by a dot:

In [12]:
sparql.setQuery("""
    SELECT ?film ?actor
    WHERE { ?film dbo:starring dbr:Kevin_Bacon . ?film dbo:starring ?actor}
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,film,actor
0,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Elijah_Wood
1,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Kevin_Bacon
2,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Julia_Davis
3,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Taylour_Paige
4,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Jacob_Tremblay
...,...,...
412,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Robert_Duvall
413,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Billy_Bob_Thornton
414,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Katherine_LaNasa
415,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Frances_O'Connor


You should now get a table with films that Kevin Bacon played in, and the actors who played with him in those films. This list also includes Kevin Bacon in each of those films.

5. To remove Kevin Bacon himself from the list of co-actors, you can use a Regular Expression to remove any actor containing the name `Kevin Bacon`, like this:

In [13]:
sparql.setQuery("""
    SELECT ?film ?actor
    WHERE { ?film dbo:starring dbr:Kevin_Bacon . ?film dbo:starring ?actor .
            FILTER (!regex(?actor, "Kevin_Bacon"))}
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,film,actor
0,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Elijah_Wood
1,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Julia_Davis
2,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Taylour_Paige
3,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Jacob_Tremblay
4,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Jonny_Coyne
...,...,...
338,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Robert_Duvall
339,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Billy_Bob_Thornton
340,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Katherine_LaNasa
341,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Frances_O'Connor


Or even better, using the semantic entity name directly:

In [14]:
sparql.setQuery("""
    SELECT ?film ?actor
    WHERE { ?film dbo:starring dbr:Kevin_Bacon . ?film dbo:starring ?actor .
            FILTER ( ?actor != dbr:Kevin_Bacon )}
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,film,actor
0,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Elijah_Wood
1,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Julia_Davis
2,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Taylour_Paige
3,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Jacob_Tremblay
4,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Jonny_Coyne
...,...,...
338,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Robert_Duvall
339,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Billy_Bob_Thornton
340,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Katherine_LaNasa
341,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Frances_O'Connor


6. To get a list of only the actors, remove `?film` from the selection. Use `SELECT DISTINCT` if needed to avoid duplicates.  Now can you get a count of the number of actors in the list?

In [18]:
sparql.setQuery("""
    SELECT DISTINCT ?actor
    WHERE { ?film dbo:starring dbr:Kevin_Bacon . ?film dbo:starring ?actor .
            FILTER ( ?actor != dbr:Kevin_Bacon )}
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,actor
0,http://dbpedia.org/resource/Elijah_Wood
1,http://dbpedia.org/resource/Julia_Davis
2,http://dbpedia.org/resource/Taylour_Paige
3,http://dbpedia.org/resource/Jacob_Tremblay
4,http://dbpedia.org/resource/Jonny_Coyne
...,...
305,http://dbpedia.org/resource/Robert_Duvall
306,http://dbpedia.org/resource/Billy_Bob_Thornton
307,http://dbpedia.org/resource/Katherine_LaNasa
308,http://dbpedia.org/resource/Frances_O'Connor


7. We'll expand the SPARQL query to get actors who co-starred with actors who co-starred with Kevin Bacon, i.e. actors who are two steps away from Kevin Bacon. First get all films that the co-stars of Kevin Bacon played in:

In [19]:
sparql.setQuery("""
    SELECT DISTINCT ?film1 ?actor1 ?film2
    WHERE { ?film1 dbo:starring dbr:Kevin_Bacon . ?film1 dbo:starring ?actor1
               . ?film2 dbo:starring ?actor1 .
               FILTER ( ?actor1 != dbr:Kevin_Bacon )  }
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,film1,actor1,film2
0,http://dbpedia.org/resource/They/Them_(film),http://dbpedia.org/resource/Anna_Chlumsky,http://dbpedia.org/resource/Bride_Hard
1,http://dbpedia.org/resource/They/Them_(film),http://dbpedia.org/resource/Anna_Chlumsky,http://dbpedia.org/resource/Gold_Diggers:_The_...
2,http://dbpedia.org/resource/They/Them_(film),http://dbpedia.org/resource/Anna_Chlumsky,http://dbpedia.org/resource/My_Girl_2
3,http://dbpedia.org/resource/They/Them_(film),http://dbpedia.org/resource/Anna_Chlumsky,http://dbpedia.org/resource/Inventing_Anna
4,http://dbpedia.org/resource/They/Them_(film),http://dbpedia.org/resource/Anna_Chlumsky,http://dbpedia.org/resource/Veep
...,...,...,...
9995,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Jacob_Tremblay,http://dbpedia.org/resource/Ciao_Alberto
9996,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Jacob_Tremblay,http://dbpedia.org/resource/Donald_Trump's_The...
9997,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Jacob_Tremblay,http://dbpedia.org/resource/Wonder_(film)__Won...
9998,http://dbpedia.org/resource/The_Toxic_Avenger_...,http://dbpedia.org/resource/Jacob_Tremblay,http://dbpedia.org/resource/The_Life_of_Chuck


8. Next we add the co-stars of the co-stars of Kevin Bacon as `?actor2`:

In [21]:
sparql.setQuery("""
    SELECT ?film1 ?actor1 ?film2 ?actor2
    WHERE { ?film1 dbo:starring dbr:Kevin_Bacon . ?film1 dbo:starring ?actor1
           . ?film2 dbo:starring ?actor1 . ?film2 dbo:starring ?actor2 .
               FILTER (?actor1 != dbr:Kevin_Bacon && ?actor2 != dbr:Kevin_Bacon ) }
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,film1,actor1,film2,actor2
0,http://dbpedia.org/resource/Tremors_(1990_film),http://dbpedia.org/resource/Michael_Gross_(actor),http://dbpedia.org/resource/Chosen_Family_(film),http://dbpedia.org/resource/Michael_Gross_(actor)
1,http://dbpedia.org/resource/Tremors_(1990_film),http://dbpedia.org/resource/Michael_Gross_(actor),http://dbpedia.org/resource/Chosen_Family_(film),http://dbpedia.org/resource/Julia_Stiles
2,http://dbpedia.org/resource/Tremors_(1990_film),http://dbpedia.org/resource/Michael_Gross_(actor),http://dbpedia.org/resource/Chosen_Family_(film),http://dbpedia.org/resource/Heather_Graham
3,http://dbpedia.org/resource/Tremors_(1990_film),http://dbpedia.org/resource/Michael_Gross_(actor),http://dbpedia.org/resource/Chosen_Family_(film),http://dbpedia.org/resource/John_Brotherton
4,http://dbpedia.org/resource/Tremors_(1990_film),http://dbpedia.org/resource/Michael_Gross_(actor),http://dbpedia.org/resource/Chosen_Family_(film),http://dbpedia.org/resource/Andrea_Savage
...,...,...,...,...
9995,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Robert_Patrick,http://dbpedia.org/resource/Future_Hunters,http://dbpedia.org/resource/Robert_Patrick
9996,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Robert_Patrick,http://dbpedia.org/resource/Future_Hunters,http://dbpedia.org/resource/Richard_Norton_(ac...
9997,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Robert_Patrick,http://dbpedia.org/resource/Future_Hunters,http://dbpedia.org/resource/Ed_Crick
9998,http://dbpedia.org/resource/Jayne_Mansfield's_Car,http://dbpedia.org/resource/Robert_Patrick,http://dbpedia.org/resource/Future_Hunters,http://dbpedia.org/resource/Linda_Carol


9. How many actors are within two steps of Kevin Bacon?  Are there no duplicates? Note the difference between `COUNT(DISTINCT ?film)` and `COUNT(?film)`.

In [22]:
#WITHOUT DUPLICATES
sparql.setQuery("""
SELECT (COUNT(DISTINCT ?actor2) AS ?actorCount)
WHERE {
  #films Kevin Bacon starred in and his direct costars
  ?film1 dbo:starring dbr:Kevin_Bacon .
  ?film1 dbo:starring ?actor1 .

  #films those costars starred in and THEIR costars (?actor2)
  ?film2 dbo:starring ?actor1 .
  ?film2 dbo:starring ?actor2 .

  # Filter out Kevin Bacon himself so we only count other people
  FILTER (?actor1 != dbr:Kevin_Bacon)
  FILTER (?actor2 != dbr:Kevin_Bacon)
}
""")


result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,actorCount
0,12939


In [23]:
#WITH DUPLICATES
sparql.setQuery("""
SELECT (COUNT(?actor2) AS ?total)
WHERE {
  ?film1 dbo:starring dbr:Kevin_Bacon .
  ?film1 dbo:starring ?actor1 .
  ?film2 dbo:starring ?actor1 .
  ?film2 dbo:starring ?actor2 .
  FILTER (?actor1 != dbr:Kevin_Bacon && ?actor2 != dbr:Kevin_Bacon)
}
""")


result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,total
0,80238


**Answer:** By running and comparing the two scripts we can see how not accounting the duplicates would result in the count being almost 5 times larger (80k VS 12K actors) than the unique actor list which is much more helpful.

# 4. Wanderlust

DBpedia is great to wander around---just like browsing through Wikipedia---but then with powerful aggregation tools at your finger tips.   Follow this walk, and make your own walks.

## Example Walk

1. Starting is always hard, so let's start with Google "dbpedia rembrandt", as we did in the lecture.

This gives you quite some info, and reveals Rembrandt is a dbpedia resource, hence `dbr:Rembrandt` (shorthand for http://dbpedia.org/resource/Rembrandt) is the unique ID inside dbpedia.

2. Let's see what information there is with `dbr:Rembrandt`

In [24]:
sparql.setQuery("""
    SELECT ?p ?o WHERE { dbr:Rembrandt ?p ?o }
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df.head(30) #Show the first 30 results

,p,o
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/2002/07/owl#Thing
1,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://xmlns.com/foaf/0.1/Person
2,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://dbpedia.org/ontology/Person
3,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://dbpedia.org/ontology/Person
4,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.ontologydesignpatterns.org/ont/dul/...
5,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.wikidata.org/entity/Q19088
6,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.wikidata.org/entity/Q215627
7,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.wikidata.org/entity/Q483501
8,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.wikidata.org/entity/Q5
9,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.wikidata.org/entity/Q729


3. The list is partial, but one relation to explore could be the "type" of entity.

In [25]:
sparql.setQuery("""
    SELECT ?o WHERE { dbr:Rembrandt rdf:type ?o }
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")

4. That's a lot---but probably you knew already a lot about him, let's move to another entity that is well-represented in Wikipedia.

In [26]:
sparql.setQuery("""
    SELECT ?o WHERE { dbr:Darth_Vader rdf:type ?o }
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")

5. With so many things, what is his actual occupation?

In [27]:
sparql.setQuery("""
    SELECT ?occupation WHERE { dbr:Darth_Vader dbo:occupation ?occupation }
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,occupation
0,http://dbpedia.org/resource/Jedi


6. Wow, but who else then is a Jedi?

In [28]:
sparql.setQuery("""
    SELECT ?person WHERE { ?person dbo:occupation dbr:Jedi }
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,person
0,http://dbpedia.org/resource/Asajj_Ventress
1,http://dbpedia.org/resource/Ahsoka_Tano
2,http://dbpedia.org/resource/Count_Dooku
3,http://dbpedia.org/resource/Darth_Vader
4,http://dbpedia.org/resource/General_Grievous
5,http://dbpedia.org/resource/Jacen_Solo
6,http://dbpedia.org/resource/Luke_Skywalker
7,http://dbpedia.org/resource/Mara_Jade
8,http://dbpedia.org/resource/Rey_(Star_Wars)
9,http://dbpedia.org/resource/Kylo_Ren__Kylo_Ren__1


7. And who is actually a Sith (spoiler alert)?  

In [29]:
sparql.setQuery("""
    SELECT ?person WHERE { ?person dbo:occupation dbr:Sith }
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,person
0,http://dbpedia.org/resource/Asajj_Ventress
1,http://dbpedia.org/resource/Count_Dooku
2,http://dbpedia.org/resource/Palpatine


8. For those who don't want to know, how many are there?

In [30]:
sparql.setQuery("""
    SELECT count(?person) WHERE { ?person dbo:occupation dbr:Sith }
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,callret-0
0,3


9. With Jedi becoming a career opportunity, what occupations are there anyway (incomplete list)?

In [31]:
sparql.setQuery("""
    SELECT ?person ?occupation WHERE { ?person dbo:occupation ?occupation } ORDER BY ?occupation
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,person,occupation
0,http://dbpedia.org/resource/Everett_Alvarez_Jr.,http://alvarezit.com/
1,http://dbpedia.org/resource/Fred_Hassan,http://caretgroup.com/
2,http://dbpedia.org/resource/Lisa_Poppaw,http://childsafecolorado.org/Staff.html
3,http://dbpedia.org/resource/$Libra_cryptocurre...,http://dbpedia.org/resource/$Libra_cryptocurre...
4,http://dbpedia.org/resource/$Libra_cryptocurre...,http://dbpedia.org/resource/$Libra_cryptocurre...
...,...,...
9995,http://dbpedia.org/resource/Dick_Foran,http://dbpedia.org/resource/Actor
9996,http://dbpedia.org/resource/Dominique_Farrugia,http://dbpedia.org/resource/Actor
9997,http://dbpedia.org/resource/Dorsey_Wright,http://dbpedia.org/resource/Actor
9998,http://dbpedia.org/resource/Farid_Shawqi,http://dbpedia.org/resource/Actor


10. Impressive, but how many people are there in DBpedia anyway, when looking at nationality?

In [32]:
sparql.setQuery("""
    SELECT count(?nationality) AS ?count ?nationality
       WHERE { ?person dbo:nationality ?nationality } ORDER BY ?nationality
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,count,nationality
0,1,http://dbpedia.org/resource/18th-century_histo...
1,1,http://dbpedia.org/resource/1936_Constitution_...
2,1,http://dbpedia.org/resource/1951_French_Grand_...
3,1,http://dbpedia.org/resource/1960_VII_South_Afr...
4,2,http://dbpedia.org/resource/1962_British_Grand...
...,...,...
2811,1,http://dbpedia.org/resource/Åland
2812,1,http://dbpedia.org/resource/Éire
2813,8,http://dbpedia.org/resource/Úrvalsdeild_karla_...
2814,1,http://dbpedia.org/resource/Úrvalsdeild_kvenna...


11. Hmm. Some interesting nationalities there... Also this may be a somewhat biased world view---let's sort that by "impact on Wikipedia"?

In [33]:
sparql.setQuery("""
    SELECT count(?nationality) AS ?count ?nationality
       WHERE { ?person dbo:nationality ?nationality } ORDER BY DESC(?count)
""")

result = sparql.query().convert().decode("utf-8")
df = pd.read_csv(StringIO(result), sep=",")
df

,count,nationality
0,14734,http://dbpedia.org/resource/United_States
1,5700,http://dbpedia.org/resource/United_Kingdom
2,4236,http://dbpedia.org/resource/India
3,3863,http://dbpedia.org/resource/Americans
4,3091,http://dbpedia.org/resource/Australia
...,...,...
2811,1,http://dbpedia.org/resource/Hollow_Water_First...
2812,1,http://dbpedia.org/resource/Indians_in_Switzer...
2813,1,http://dbpedia.org/resource/Cinema_of_Mexico
2814,1,http://dbpedia.org/resource/Peru_at_the_2016_S...


12. Etcetera. Etcetera. Etcetera. ...

**Part 3**

Are there issues with completeness of encoding (does Rembrandt have an occupation or nationality?) or with selection and bias/representation that you observed in this example walk? Take at least a paragraph or two to reflect on the results.

**Answer**: Yes, there are clear issues with completeness of encoding. In the Rembrandt query, DBpedia identifies him as a Person and Artist, but the visible results do not directly show a simple dbo:occupation or dbo:nationality value. 
Additionally, there are also issues with selection, bias, and representation. For example, the nationality query shows that some “nationalities” are not countries or national identities at all, and when sorted by count, the United States and United Kingdom dominate the list. This likely reflects what is most represented in Wikipedia/DBpedia rather than a neutral and unbiased representation of the world.
Therefore, we can suggest that the example walk may not be entirely representative, neutral, or automatically reliable. We should adopt a critical approach when analyzing semantic data. 

## Make your own walk!

In a similar way as the example walk, make your own walk through DBpedia/Wikipedia. Explore some of the amazing power SPARQL queries give you to explore, as well as being aware of the limitations and bias of the collection and the encoding. We reward creativity as much as technical skills: some of the most interesting queries are a very simple SPARQL statement! Just like in Assignment 1, please do put a topic that you are interested into this one, not one of our boring example ones.

> NOTE: Recall that finding the exact way of encoding of the *Subject-Predicate-Object* triplets and (in particular) knowing the IDs of entities can be hard.  But searching for `dbpedia entity-by-name` in your favorite search engine will help a lot!  (For more serious applications, you would use a range of knowledge graphs and tools like https://qlever.dev/ which offer relation and entity completion in special interfaces.)



**(Your answers with query code blocks and text block comments here)**

# 5. Annotating a corpus

Up to now, we have only queried DBpedia/Wikipedia, but the true power of linked open data is the ability to connect any corpus to the entities in DBpedia.  Manually annotating a corpus is very laborious, but automatic tools for entity linking can potentially annotate any DBpedia entity found in any text.

In Python, Spacy has a DBpedia Spotlight-based entity recognizer. Spacy is a very useful Python tool that can handle a large variety of text processing tasks, including also named entity recognition, sentiment analysis, part-of-speech tagging or text categorization, for various languages. It is definitely worth exploring more in your project or some other time.

As this assignment is already long enough, we will only briefly show how it works and ask you to experiment with it in a free format.

Let's install it:

In [1]:
#!pip install spacy-dbpedia-spotlight

We will make a text processing pipeline that only includes the DBedia entity linker, and nothing else. This may take a minute:

In [2]:
import spacy_dbpedia_spotlight

nlp = spacy_dbpedia_spotlight.create('en')

print(nlp.pipe_names)


['dbpedia_spotlight']


If you are able to experiment with a different language than English, we encourage you to try it! For that, you can change the 'en' language code above to a different language code.

Now, we simply need to use the nlp function on a string, and it will attempt to recognize DBPedia entities in it:

In [3]:
doc = nlp('The University Of Amsterdam is a Dutch higher education institution located in Amsterdam.')

print(doc.ents) #This just prints the entities that were found

#This prints some more details, including the DBPedia identifier and the similarity score:
print([(ent.text, ent.kb_id_, ent._.dbpedia_raw_result['@similarityScore']) for ent in doc.ents])

(University Of Amsterdam, Amsterdam)
[('University Of Amsterdam', 'http://dbpedia.org/resource/University_of_Amsterdam', '0.9997770213964399'), ('Amsterdam', 'http://dbpedia.org/resource/Amsterdam', '0.999936852788097')]


Let's just make a tiny change to the input and see if the output changes:

In [4]:
doc = nlp('The University of Amsterdam is a Dutch higher education institution located in Amsterdam.')
print([(ent.text, ent.kb_id_, ent._.dbpedia_raw_result['@similarityScore']) for ent in doc.ents])

[('Amsterdam', 'http://dbpedia.org/resource/Amsterdam', '0.999936852788097'), ('Amsterdam', 'http://dbpedia.org/resource/Amsterdam', '0.999936852788097')]


Interesting: it no longer recognizes the university in my case.

Let's try a longer text: here's a recent news item (https://www.bbc.com/future/article/20260414-the-monkey-selfie-that-predicted-the-ai-age ).

In [5]:
doc = nlp("This monkey selfie will protect you from AI slop. \n What happens when something that isn't human makes art? A series of bizarre court battles trying to answer that question centred around this image. Ultimately, it will influence what ends up on your screens and headphones forever.\n It was a humid day in the Indonesian jungle, and photographer David Slater was following a group of crested black macaques, a critically endangered and particularly photogenic species of monkey. He wanted pictures, but the macaques were nervous. So, Slater put his camera on a tripod with autofocus on and a flashbulb, allowing the monkeys to inspect it. Just as he hoped, they started playing with his gear. Then one of them reached up and hit the shutter button while staring directly into the lens. The result was a selfie, taken by a monkey. And its toothy grin inadvertently answered a basic question that sits at the heart of technology. What came next was nearly a decade of legal battles around an unusual dispute: when something that isn't human makes a work of art, who owns the copyright? Thanks to AI, that's become an issue with some deep implications for modern life – and what it means to be human. One of the most alarming predictions about AI is that corporations will replace the human-created music, movies and books you love with an endless stream of AI slop. But the US Supreme Court just upheld a decision about AI and copyright which suggests that future may be harder to pull off than the tech industry hoped. The path is still uncertain, and right now, the legal system is the site of a battle that will shape what you read, watch and listen to for the rest of your life. It all traces back to that one little monkey.")
print([(ent.text, ent.kb_id_, ent._.dbpedia_raw_result['@similarityScore']) for ent in doc.ents])

[('monkey selfie', 'http://dbpedia.org/resource/Monkey_selfie_copyright_dispute', '1.0'), ('AI', 'http://dbpedia.org/resource/Artificial_intelligence', '0.9999999903391199'), ('court', 'http://dbpedia.org/resource/Court', '0.9999476047323947'), ('headphones', 'http://dbpedia.org/resource/Headphones', '0.9999966268935668'), ('Indonesian', 'http://dbpedia.org/resource/Indonesian_language', '0.9915499659701658'), ('jungle', 'http://dbpedia.org/resource/Jungle', '0.9855265791518633'), ('David Slater', 'http://dbpedia.org/resource/David_Slater', '0.9999999999997726'), ('critically endangered', 'http://dbpedia.org/resource/Endangered_species', '0.8501348084272634'), ('species', 'http://dbpedia.org/resource/Species', '0.9999995679148493'), ('tripod', 'http://dbpedia.org/resource/Tripod', '0.9999999797498733'), ('autofocus', 'http://dbpedia.org/resource/Autofocus', '1.0'), ('shutter', 'http://dbpedia.org/resource/Focal-plane_shutter', '0.9999965219045867'), ('lens', 'http://dbpedia.org/resourc

Did it correctly identify entities (rather than strings of text) in the text?  How do the detected entities cover the main content?

**Answer**: For the University of Amsterdam, the annotation is accurate and meaningful: it is recognized as a specific institution rather than just a sequence of words. This indicates that the system successfully links the text to a real-world entity, which contributes directly to understanding the context of the corpus (e.g., an academic or institutional setting). 

However, the example of the monkey selfie reveals how he annotation struggles to consistently capture it as a coherent entity. It may treat parts of the phrase separately or link it in a way that does not fully reflect its significance. This suggests that the system is less effective when dealing with more context-dependent or culturally specific references. 

The output demonstrates that the system reflects most of the central topics of the text. Overall, The University of Amsterdam is well-identified and contributes to the main narrative, whereas the monkey selfie example is only partially and not accurately represented.

**Part 4**

Now, it's your turn to experiment with entity linking. Find a paragraph of text anywhere in your target language -- text with concrete names of persons, organizations, locations, events, ... (like news) may have more entities than abstract philosophy -- and try it for yourself. How well does this work?  Did it find all or most entities?  Do you see "errors"?

We will use the following news article from BBC: https://www.bbc.com/news/articles/crm1y89vek8o

In [6]:
doc = nlp(
    "Meta will cut thousands of jobs next month as it spends more than ever on artificial intelligence (AI) projects. "
    "The company told employees in a memo on Thursday that it planned to cut 10% of its workforce - roughly 8,000 staff. "
    "It said it would also not fill thousands more open jobs it had been hiring for. "
    "A key reason for the layoffs is Meta's increased spending in other areas of the company, including AI, for which it will this year spend $135bn (£100bn). "
    "This is roughly equal to the amount it has spent on AI in the previous three years combined, according to a person who viewed the memo. "
    "A spokesman for Meta confirmed the planned job cuts but declined to comment further. "
    "Mark Zuckerberg, Meta's co-founder and chief executive, made public comments in January that essentially telegraphed the company would be cutting jobs again this year. "
    "The Meta boss said he had seen how much more productive workers who relied heavily on AI tools had become, noting a single person could now complete projects that would have previously required a large team. "
    "\"I think that 2026 is going to be the year that AI starts to dramatically change the way that we work,\" Zuckerberg said. "
    "Last week Reuters news agency reported that Meta was planning to cut potentially more than 10,000 employees this year. "
    "The memo to employees on Thursday was first reported by Bloomberg. "
    "While Meta has already cut around 2,000 workers in two smaller rounds of layoffs already this year, employees had been braced for weeks for a much deeper cut, as the BBC previously reported. "
    "Meta's spending and internal focus had shifted heavily in recent months toward catching up on the development of AI models and tools. "
    "The company just this week informed employees that it would begin tracking and logging their interactions with work computers in order to help train and improve its AI models, a move one employee called \"dystopian\" given the looming layoffs. "
    "\"This company has become obsessed with AI,\" they told the BBC."
)

print([(ent.text, ent.kb_id_, ent._.dbpedia_raw_result['@similarityScore']) for ent in doc.ents])

[('artificial intelligence', 'http://dbpedia.org/resource/Artificial_intelligence', '0.9999999940355337'), ('AI', 'http://dbpedia.org/resource/Artificial_intelligence', '0.9999999890624168'), ('AI', 'http://dbpedia.org/resource/Artificial_intelligence', '0.9999999890624168'), ('AI', 'http://dbpedia.org/resource/Artificial_intelligence', '0.9999999890624168'), ('Mark Zuckerberg', 'http://dbpedia.org/resource/Mark_Zuckerberg', '1.0'), ('AI', 'http://dbpedia.org/resource/Artificial_intelligence', '0.9999999890624168'), ('AI', 'http://dbpedia.org/resource/Artificial_intelligence', '0.9999999890624168'), ('Reuters', 'http://dbpedia.org/resource/Reuters', '0.9999999931321781'), ('Bloomberg', 'http://dbpedia.org/resource/Bloomberg_News', '0.9191828292122045'), ('BBC', 'http://dbpedia.org/resource/BBC_News', '0.9945539007759386'), ('AI', 'http://dbpedia.org/resource/Artificial_intelligence', '0.9999999890624168'), ('logging', 'http://dbpedia.org/resource/Logging', '0.9987289597408097'), ('AI',

The system generally succeeds in identifying entities, especially when it comes to concrete names of people and organizations. For example, Meta is consistently recognized as a company and Mark Zuckerberg is identified as a specific person. Organizations such as Reuters, Bloomberg, and BBC are detected as distinct entities. 

In terms of coverage, the entities reflect the main structure of the article quite well. For example, “artificial intelligence” and “AI” are consistently linked to the same DBpedia resource with extremely high scores (~0.99999999), which shows very high confidence and consistency. 
However, the output also reveals some weaker or less meaningful annotations. For example, “logging” is linked to the DBpedia page for logging (as in timber), which does not fit the context of tracking employee computer activity. 
Therefore, we can suggest that the system sometimes annotates words that are not central in the article or misinterprets their context. 

Important events such as "cut 10% of its workforce", "spend $135bn", or the broader idea of layoffs driven by AI investment are not considered entities. They remain as strings of text.  

Now assume you have a large text corpus annotated in this way (think up your own corpus of interest, or otherwise think about the corpus of movie reviews you explored earlier).   

Can you think up something you could explore using these annotations? E.g. if we remember the movie reviews collection from Assignment 2 we could look at particular actors, but also about comparing male and female actors as two groups, and doing aggregated queries about male and female actor mentions in the whole corpus of negative and positive movie reviews (similar to the SPARQL queries earlier).

In the context of the corpus of AI restructuring corporate organizations such as Meta, we could explore how AI, as a topic, is framed in news coverage. 
We would compare how AI is associated with positive VS negative framings. A positive framing could be centered on efficiency and productivity, whereas a negative framing could be about direct consequences, or a dystopian point of view. Using these annotations, we can count how often AI-related entities co-occur with terms such as "dystopian" or associated with job layoffs. Specific actors (executives, companies) will either be linked to positive productivity narratives or negative labor impacts. 

# 6. Discussion and Reflection

Feel encouraged to explore some or all the options offered by the custom search engine, but include at least one page (+/- 500 words) reflecting on how well well semantic based representations works--trying to better understand both
the strengths and limitations of using semantic representations.

In particular we can compare the relative strengths and weaknesses of plain text vs semantic representations for search and analysis, and can discuss and reflect on the diﬀerent ways to get a grip on semantics through text and plain text search, and through semantic annotations and SPARQL queries. Compare these two types of search in terms of how they model semantics and how they alter the search process. For which kinds of search tasks would each be suitable? What do you think are strengths and weaknesses of each? Is there a diﬀerence in power, control, and transparency of the user? How hard or easy is each way of querying? Would complex querying become simpler if you were using this on a
daily basis, or is there an inherent extra effort? How serious are annotation or data errors in semantic searching? Are similar errors also aﬀecting text search, but remaining less visible (so go unnoticed by searchers and researchers)? How does the semantics captured with standard keyword search and
semantic annotation diﬀer from the semantics captured by topic modeling? Etc.

Feel free to include any aspect you come across and find of relevance, and try to be concrete and include examples (just observing interesting cases is already of great value), which can make this section much longer than 1 page. No need to be conservative, and feel encouraged to give speculative
answers--each of these can be regarded as an hypothesis that could be tested in later research.

> ***NOTE**: While we permit helpful use of GenAI, under the condition of complete disclosure and full transparency, this does not apply to writing reflections.  Although GenAI may be a useful source for information, or correct your grammar and spelling, we insist that when we ask for a reflection, the actual reflection part is fully and exclusively authored by *you* about *your* learning experience.*

> *Reflections are not graded, but are very useful for yourself to understand what you learned during the assignment. They  also provide useful context for lecturers to understand what you tried or attempted to do (in particular in case it is executed imperfectly).*

> *Even though assignments like these are graded, the weight is small and meant as formative feedback, and the assignments are contructed for learning-by-doing, and learning-by-discovery. Reflection is an essential part to achieve this.*

### Report part 5 (a): Discussion/Reflection

(We left the assignment light-weight as you have done now three  different lab assignments, working with very different representation of large corpora, and different ways of exploring and analyzing these corpora. So this is a good moment to think back and reflect on this assignment, but also in comparison to your earlier experiments!)

(Note this is a reflection on what *you* learned from *your* experiments, not a general essay or rewritten version of ChatGPT's thoughts about this in general.)

## Report part 5 (b): GenAI Disclosure Section

(Rather than banning the use of GenAI tools, we demand complete disclosure and full transparency. This allows us to ensure you achieve the course's learning objectives.)

(Briefly detail any use of GenAI in the coding and the writing.  Detail the role of GenAI and your own role, and how you checked or verified the output.)  

(If you used GenAI, also briefly reflect on whether you will be able to do all the main parts yourself, and remain in control, while delegating some of the coding details to your helpful assistant.)